In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

In [26]:
mnist = fetch_openml('mnist_784', version=1, as_frame=True, parser='auto')
df = pd.concat([mnist.data, mnist.target], axis=1)
df.head()

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784,class
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,5
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,4
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,9


In [27]:
X_train, X_test, Y_train, Y_test = train_test_split(mnist.data, mnist.target.astype(int), test_size=10000, random_state=42)
print(X_train.shape)
print(X_test.shape)
print(Y_train.shape)
print(Y_test.shape)

(60000, 784)
(10000, 784)
(60000,)
(10000,)


In [28]:
X_train = X_train.T
X_test = X_test.T

# normalize input into between 0 and 1
X_train = X_train / 255
X_test = X_test / 255

In [ ]:
def initialize_param():
    """initialize the 2 layer parameters for neural network(-0.5 to center the mean to 0)"""
    w1 = np.random.rand(10, 784) - 0.5
    b1 = np.random.rand(10,1) - 0.5
    w2 = np.random.rand(10, 10) - 0.5
    b2 = np.random.rand(10, 1) - 0.5
    return w1, b1, w2, b2

def ReLU(X):
    return np.maximum(X, 0)

def Softmax(Z):
    Z = Z - np.max(Z, axis=0, keepdims=True)
    return np.exp(Z) / np.sum(np.exp(Z), axis=0)

def forward_prop(w1, b1, w2, b2, X):
    Z1 = np.dot(w1, X) + b1
    A1 = ReLU(Z1)
    Z2 = np.dot(w2, A1) + b2
    A2 = Softmax(Z2)
    return Z1, A1, Z2, A2

def one_hot_converter(Y):
    one_hot_Y = np.zeros((Y.size, Y.max()+1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def backward_prop(w1, b1, w2, b2, Z1, A1, Z2, A2, X, Y):
    m = Y.size

    Y = one_hot_converter(Y)
    dZ2 = A2 - Y
    dw2 = 1/m * np.dot(dZ2, A1.T)
    db2 = 1/m * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = np.dot(w2.T, dZ2) * (Z1 > 0)
    dw1 = 1/m * np.dot(dZ1, X.T)
    db1 = 1/m * np.sum(dZ1, axis=1, keepdims=True)
    return dw1, db1, dw2, db2

def update_params(w1, b1, w2, b2, dw1, db1, dw2, db2, learning_rate=0.1):
    w1 = w1 - learning_rate*dw1
    b1 = b1 - learning_rate*db1
    w2 = w2 - learning_rate*dw2
    b2 = b2 - learning_rate*db2
    return w1, b1, w2, b2

def get_predictions(A2):
  return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
  return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, epoch, learning_rate=0.1):
    w1, b1, w2, b2 = initialize_param()

    for i in range(epoch):
        Z1, A1, Z2, A2 = forward_prop(w1, b1, w2, b2, X)
        dw1, db1, dw2, db2 = backward_prop(w1, b1, w2, b2, Z1, A1, Z2, A2, X, Y)
        w1, b1, w2, b2 = update_params(w1, b1, w2, b2, dw1, db1, dw2, db2, learning_rate)

        if (i%20) == 0:
            print("Iteration number: ", i)
            print("Accuracy: ", get_accuracy(get_predictions(A2), Y))

    return w1, b1, w2, b2

In [30]:
W1, B1, W2, B2 = gradient_descent(X_train, Y_train, 1000, 0.1)

Iteration number:  0
Accuracy:  0.09515
Iteration number:  20
Accuracy:  0.29141666666666666
Iteration number:  40
Accuracy:  0.39676666666666666
Iteration number:  60
Accuracy:  0.5011333333333333
Iteration number:  80
Accuracy:  0.5884833333333334
Iteration number:  100
Accuracy:  0.6485
Iteration number:  120
Accuracy:  0.6923333333333334
Iteration number:  140
Accuracy:  0.7250166666666666
Iteration number:  160
Accuracy:  0.7497333333333334
Iteration number:  180
Accuracy:  0.7677
Iteration number:  200
Accuracy:  0.7823166666666667
Iteration number:  220
Accuracy:  0.7936833333333333
Iteration number:  240
Accuracy:  0.8033333333333333
Iteration number:  260
Accuracy:  0.8110833333333334
Iteration number:  280
Accuracy:  0.8183
Iteration number:  300
Accuracy:  0.8252333333333334
Iteration number:  320
Accuracy:  0.8306333333333333
Iteration number:  340
Accuracy:  0.83545
Iteration number:  360
Accuracy:  0.8399166666666666
Iteration number:  380
Accuracy:  0.8432666666666667
It

In [31]:
Z1test, A1test, Z2test, A2test = forward_prop(W1, B1, W2, B2, X_test)
test_acc = get_accuracy(get_predictions(A2test), Y_test)
print("test accuracy: ", test_acc)

test accuracy:  0.8816
